In [21]:
!pip install pymupdf faiss-cpu sentence-transformers

In [24]:
# ---------------------------------------------------------------------------
# PDF → text (Bulletproof version)
# ---------------------------------------------------------------------------
def iter_pdf_chunks(path: Path) -> Generator[dict, None, None]:
    print(f"    [DEBUG] Opening PDF: {path.name}", flush=True)
    doc = fitz.open(path)
    chunk_id = 0
    total_pages = doc.page_count

    # Use a standard range loop to ensure we touch every page
    for page_num in range(total_pages):
        page = doc[page_num]
        text = page.get_text().strip()

        # Debug: Print every 5 pages so we know it's moving
        if page_num % 5 == 0 or page_num == total_pages - 1:
            print(f"    [DEBUG] Processing page {page_num + 1}/{total_pages}...", flush=True)

        if not text:
            continue

        start = 0
        length = len(text)

        while start < length:
            end = min(start + CHUNK_SIZE, length)

            if end < length:
                boundary = max(
                    text.rfind(". ", start, end),
                    text.rfind("\n", start, end),
                    text.rfind("! ", start, end),
                    text.rfind("? ", start, end),
                )
                if boundary > start + CHUNK_SIZE // 2:
                    end = boundary + 1

            chunk = text[start:end].strip()

            if len(chunk) > 40:
                yield {
                    "text": chunk,
                    "source": path.name,
                    "chunk_id": chunk_id,
                }
                chunk_id += 1

            if end >= length:
                break
            start = end - CHUNK_OVERLAP

    doc.close()

# ---------------------------------------------------------------------------
# Main Execution Function (Fixed Warning)
# ---------------------------------------------------------------------------
def run_indexing(force: bool = False) -> None:
    print("\n" + "="*58)
    print("  RAG Index Builder — Expert Mode")
    print("="*58)

    DOCS_FOLDER.mkdir(parents=True, exist_ok=True)
    INDEX_FOLDER.mkdir(parents=True, exist_ok=True)

    pdf_files = sorted(DOCS_FOLDER.glob("*.pdf"))
    if not pdf_files:
        print(f"\n[!] No PDFs found in {DOCS_FOLDER}")
        return

    stored_hashes = {} if force else load_hashes()
    cached_chunks = []

    if not force and CHUNKS_FILE.exists():
        print("[DEBUG] Loading existing chunks...", flush=True)
        with open(CHUNKS_FILE, "rb") as f:
            cached_chunks = pickle.load(f)
        print(f"[Cache] {len(cached_chunks)} chunks loaded.\n")

    current_hashes = {}
    new_pdfs = []

    for pdf in pdf_files:
        digest = md5_of_file(pdf)
        current_hashes[pdf.name] = digest
        if force or stored_hashes.get(pdf.name) != digest:
            new_pdfs.append(pdf)
            print(f"[New/Changed] {pdf.name}")
        else:
            print(f"[Unchanged]   {pdf.name}")

    if not new_pdfs:
        print("\n[RAG] Everything up to date.")
        return

    print(f"\n[DEBUG] Initializing Model: {EMBEDDING_MODEL}...", flush=True)
    model = SentenceTransformer(EMBEDDING_MODEL)

    # FIXED: Updated method name to avoid FutureWarning
    dim = model.get_embedding_dimension()
    print(f"[DEBUG] Model loaded. Dimension: {dim}", flush=True)

    if INDEX_FILE.exists() and not force:
        print("[DEBUG] Loading existing FAISS index...", flush=True)
        index = faiss.read_index(str(INDEX_FILE))
    else:
        index = faiss.IndexFlatL2(dim)
        print(f"[DEBUG] Created new FAISS index.")

    new_sources = {p.name for p in new_pdfs}
    all_chunks = [c for c in cached_chunks if c["source"] not in new_sources]

    total_new = 0
    for pdf_path in new_pdfs:
        print(f"\n[+] Starting: {pdf_path.name}")
        start_time = time.time()

        added = stream_encode(
            iter_pdf_chunks(pdf_path),
            model,
            index,
            all_chunks
        )

        elapsed = time.time() - start_time
        total_new += added
        print(f"    ✓ {added} chunks added (Time: {elapsed:.2f}s)")
        gc.collect()

    faiss.write_index(index, str(INDEX_FILE))
    with open(CHUNKS_FILE, "wb") as f:
        pickle.dump(all_chunks, f)
    save_hashes(current_hashes)

    print("\n" + "="*58)
    print("DONE")
    print(f"Total vectors in index: {index.ntotal}")
    print("="*58)

# Run with force=True to fix the previous empty run
run_indexing(force=True)


  RAG Index Builder — Expert Mode
[New/Changed] 2112.10752v2.pdf
[New/Changed] 2212.09748v2.pdf
[New/Changed] 2307.01952v1.pdf

[DEBUG] Initializing Model: all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[DEBUG] Model loaded. Dimension: 384
[DEBUG] Created new FAISS index.

[+] Starting: 2112.10752v2.pdf
    [DEBUG] Opening PDF: 2112.10752v2.pdf
MuPDF error: syntax error: no XObject subtype specified

MuPDF error: syntax error: no XObject subtype specified

MuPDF error: syntax error: no XObject subtype specified

MuPDF error: syntax error: no XObject subtype specified

MuPDF error: syntax error: no XObject subtype specified

MuPDF error: syntax error: no XObject subtype specified

MuPDF error: syntax error: no XObject subtype specified

MuPDF error: syntax error: no XObject subtype specified

MuPDF error: syntax error: no XObject subtype specified

MuPDF error: syntax error: no XObject subtype specified

    [DEBUG] Processing page 1/45...
    [DEBUG] Processing page 6/45...
    [DEBUG] Processing page 11/45...
    [DEBUG] Processing page 16/45...
    [DEBUG] Processing page 21/45...
    [DEBUG] Processing page 26/45...
    [DEBUG] Processing page 31/45...
    [DEBUG] Processing page 3

In [26]:
import pickle

# Path to your chunks
CHUNKS_FILE = "index/chunks.pkl"

with open(CHUNKS_FILE, "rb") as f:
    chunks = pickle.load(f)

print(f"Total chunks in index: {len(chunks)}")
print("-" * 30)

# Show the first 3 chunks
for i, chunk in enumerate(chunks[:3]):
    print(f"CHUNK #{i}")
    print(f"Source: {chunk['source']}")
    print(f"Text snippet: {chunk['text'][:150]}...")
    print("-" * 30)

Total chunks in index: 168
------------------------------
CHUNK #0
Source: 2112.10752v2.pdf
Text snippet: High-Resolution Image Synthesis with Latent Diffusion Models
Robin Rombach1 *
Andreas Blattmann1 ∗
Dominik Lorenz1
Patrick Esser
Bj¨orn Ommer1
1Ludwig...
------------------------------
CHUNK #1
Source: 2112.10752v2.pdf
Text snippet: state-of-the-art synthesis results on
image data and beyond. Additionally, their formulation al-
lows for a guiding mechanism to control the image gen...
------------------------------
CHUNK #2
Source: 2112.10752v2.pdf
Text snippet: resources while retaining their quality and ﬂexibility, we
apply them in the latent space of powerful pretrained au-
toencoders. In contrast to previo...
------------------------------


In [27]:
import pickle

# Path to your chunks
CHUNKS_FILE = "index/chunks.pkl"

with open(CHUNKS_FILE, "rb") as f:
    chunks = pickle.load(f)

print(f"Total chunks in index: {len(chunks)}")
print("-" * 30)

# Show the first 3 chunks
for i, chunk in enumerate(chunks[:3]):
    print(f"CHUNK #{i}")
    print(f"Source: {chunk['source']}")
    print(f"Text snippet: {chunk['text'][:150]}...")
    print("-" * 30)

Total chunks in index: 168
------------------------------
CHUNK #0
Source: 2112.10752v2.pdf
Text snippet: High-Resolution Image Synthesis with Latent Diffusion Models
Robin Rombach1 *
Andreas Blattmann1 ∗
Dominik Lorenz1
Patrick Esser
Bj¨orn Ommer1
1Ludwig...
------------------------------
CHUNK #1
Source: 2112.10752v2.pdf
Text snippet: state-of-the-art synthesis results on
image data and beyond. Additionally, their formulation al-
lows for a guiding mechanism to control the image gen...
------------------------------
CHUNK #2
Source: 2112.10752v2.pdf
Text snippet: resources while retaining their quality and ﬂexibility, we
apply them in the latent space of powerful pretrained au-
toencoders. In contrast to previo...
------------------------------


In [28]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Load the model and index
model = SentenceTransformer("all-MiniLM-L6-v2")
index = faiss.read_index("index/faiss.index")
with open("index/chunks.pkl", "rb") as f:
    all_chunks = pickle.load(f)

def test_search(query, k=3):
    # 2. Encode the question
    query_vec = model.encode([query]).astype("float32")

    # 3. Search the index
    distances, indices = index.search(query_vec, k)

    print(f"\nQUERY: {query}")
    print("="*50)

    for i, idx in enumerate(indices[0]):
        if idx == -1: continue # No match found

        chunk = all_chunks[idx]
        print(f"MATCH {i+1} (Score: {distances[0][i]:.4f})")
        print(f"FROM: {chunk['source']}")
        print(f"CONTENT: {chunk['text']}")
        print("-" * 50)

# 4. Try a search!
# If you used the test_college_advisor.pdf, try this:
test_search("What is the main advantage of applying diffusion models in latent space instead of pixel space?")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



QUERY: What is the main advantage of applying diffusion models in latent space instead of pixel space?
MATCH 1 (Score: 0.5790)
FROM: 2307.01952v1.pdf
CONTENT: s for the Stable Diffusion architecture. These are modular,
and can be used individually or together to extend any model. Although the following strategies are
implemented as extensions to latent diffusion models (LDMs) [38], most of them are also applicable
to their pixel-space counterparts.
Figure 1: Left: Comparing user preferences between SDXL and Stable Diffusion 1.5 & 2.1. While SDXL already
--------------------------------------------------
MATCH 2 (Score: 0.8378)
FROM: 2212.09748v2.pdf
CONTENT: transformer architecture. We train latent diffusion models
of images, replacing the commonly-used U-Net backbone
with a transformer that operates on latent patches. We an-
alyze the scalability of our Diffusion Transformers (DiTs)
through the lens of forward pass complexity as measured by
Gﬂops. We ﬁnd that DiTs with higher Gﬂops—

In [18]:
!rm -rf index/